In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'Switch runtime to GPU'
print('GPU :', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

Mounted at /content/drive
GPU : Tesla T4
VRAM: 15.6 GB


In [2]:
!pip install -q pytorch-fid piq sewar scikit-image tqdm opencv-python-headless \
               huggingface_hub safetensors fvcore

import os
if not os.path.exists('/content/AdaFace'):
    !git clone -q https://github.com/mk-minchul/AdaFace.git /content/AdaFace
    print('AdaFace repo cloned.')
else:
    print('AdaFace repo already present.')

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 11.5 MB/s eta 0:00:00
AdaFace repo cloned.


In [3]:
import os, sys, shutil

PROJECT    = '/content/drive/MyDrive/Sketch2Face'
SRC_DIR    = PROJECT
DATA_DIR   = os.path.join(PROJECT, 'dataset', 'split')
OUTPUT_DIR = os.path.join(PROJECT, 'output_pix2pix_adaface')
ADAFACE_DIR= os.path.join(PROJECT, 'pretrained', 'adaface_ir50_webface4m')

for sub in ['checkpoints', 'samples', 'inference/generated', 'inference/real']:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)
os.makedirs(ADAFACE_DIR, exist_ok=True)

# Copy source files into /content
for fname in ['pix2pix.py', 'dataset.py', 'augmentation.py']:
    src = os.path.join(SRC_DIR, fname)
    dst = os.path.join('/content', fname)
    if os.path.isfile(src):
        shutil.copy(src, dst); print(f'  Copied {fname}')
    else:
        print(f'  WARNING: {src} not found')

for p in ['/content', '/content/AdaFace']:
    if p not in sys.path: sys.path.insert(0, p)
os.chdir('/content')

CFG = dict(
    n_epochs     = 200,
    decay_start  = 100,   # linear LR decay from epoch 100 onwards
    batch_size   = 4,     # T4 safe; use 2 if OOM
    num_workers  = 2,
    lr           = 2e-4,
    beta1        = 0.5,
    lambda_l1    = 100.0,
    lambda_vgg   = 10.0,
    lambda_fm    = 10.0,
    lambda_id    = 10.0,  # AdaFace — higher than before so it shapes geometry
    save_every   = 20,
    sample_every = 10,
    seed         = 42,
)
print('Config:', CFG)

  Copied pix2pix.py
  Copied dataset.py
  Copied augmentation.py
Config: {'n_epochs': 200, 'decay_start': 100, 'batch_size': 4, 'num_workers': 2, 'lr': 0.0002, 'beta1': 0.5, 'lambda_l1': 100.0, 'lambda_vgg': 10.0, 'lambda_fm': 10.0, 'lambda_id': 10.0, 'save_every': 20, 'sample_every': 10, 'seed': 42}


In [4]:
SENTINEL = os.path.join(ADAFACE_DIR, 'model.safetensors')

if os.path.isfile(SENTINEL):
    print(f'AdaFace already downloaded: {ADAFACE_DIR}')
else:
    print('Downloading AdaFace IR-50 from HuggingFace...')
    from huggingface_hub import hf_hub_download
    REPO = 'minchul/cvlface_adaface_ir50_webface4m'

    hf_hub_download(REPO, 'files.txt',
                    local_dir=ADAFACE_DIR, local_dir_use_symlinks=False)
    with open(os.path.join(ADAFACE_DIR, 'files.txt')) as f:
        extras = [l.strip() for l in f if l.strip()]

    for fname in extras + ['config.json', 'wrapper.py', 'model.safetensors']:
        dst = os.path.join(ADAFACE_DIR, fname)
        if not os.path.isfile(dst):
            print(f'  {fname}...')
            hf_hub_download(REPO, fname,
                            local_dir=ADAFACE_DIR, local_dir_use_symlinks=False)

    print('Download complete.')

assert os.path.isfile(SENTINEL)
print('AdaFace IR-50 ready.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


files.txt:   0%|          | 0.00/350 [00:00<?, ?B/s]

  README.md...


README.md: 0.00B [00:00, ?B/s]

  pretrained_model/model.pt...


...

  wrapper.py...


wrapper.py:   0%|          | 0.00/766 [00:00<?, ?B/s]

  model.safetensors...


model.safetensors:   0%|          | 0.00/175M [00:00<?, ?B/s]

Download complete.
AdaFace IR-50 ready.


In [5]:
import torch.nn as nn

def init_weights(net: nn.Module, std: float = 0.02) -> nn.Module:
    for m in net.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            w = 'weight_orig' if hasattr(m, 'weight_orig') else 'weight'
            nn.init.normal_(getattr(m, w), 0.0, std)
            if getattr(m, 'bias', None) is not None:
                nn.init.constant_(m.bias, 0.0)
        elif isinstance(m, nn.InstanceNorm2d) and m.affine:
            nn.init.normal_(m.weight, 1.0, std)
            nn.init.constant_(m.bias, 0.0)
    return net

In [6]:
import torch
import torch.nn as nn
from pix2pix import PatchDiscriminator

class MultiScaleDiscriminator(nn.Module):
    """
    Two PatchDiscriminators:
        D1 — full resolution 256x256  (global face shape)
        D2 — half resolution 128x128  (local region quality)

    Returns list of (logits, features) — one per scale.
    """
    def __init__(self, in_ch=6, ndf=64):
        super().__init__()
        self.D1 = PatchDiscriminator(in_ch, ndf)
        self.D2 = PatchDiscriminator(in_ch, ndf)
        self.pool = nn.AvgPool2d(kernel_size=3, stride=2,
                                 padding=1, count_include_pad=False)

    def forward(self, sketch, photo):
        out1 = self.D1(sketch, photo)
        out2 = self.D2(self.pool(sketch), self.pool(photo))
        return [out1, out2]

print('MultiScaleDiscriminator defined.')

MultiScaleDiscriminator defined.


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm


# ── LSGAN ────────────────────────────────────────────────────────────────────
class LSGANLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def discriminator_loss(self, real_outs, fake_outs):
        loss = torch.tensor(0.0, device=real_outs[0][0].device)
        for (r, _), (f, _) in zip(real_outs, fake_outs):
            loss = loss + 0.5 * (
                self.mse(r, torch.ones_like(r)) +
                self.mse(f, torch.zeros_like(f))
            )
        return loss / len(real_outs)

    def generator_loss(self, fake_outs):
        loss = torch.tensor(0.0, device=fake_outs[0][0].device)
        for (f, _) in fake_outs:
            loss = loss + 0.5 * self.mse(f, torch.ones_like(f))
        return loss / len(fake_outs)


# ── VGG-19 Perceptual ────────────────────────────────────────────────────────
class PerceptualLoss(nn.Module):
    _LAYERS = {'relu2_2': 9, 'relu3_3': 18}

    def __init__(self):
        super().__init__()
        vgg  = tvm.vgg19(weights=tvm.VGG19_Weights.IMAGENET1K_V1).features
        prev = 0
        self.slices = nn.ModuleList()
        for idx in sorted(self._LAYERS.values()):
            self.slices.append(nn.Sequential(*list(vgg.children())[prev:idx+1]))
            prev = idx + 1
        for p in self.parameters(): p.requires_grad_(False)
        self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

    def _pre(self, x):
        return ((x * 0.5 + 0.5).clamp(0,1) - self.mean) / self.std

    def forward(self, fake, real):
        fp, rp = self._pre(fake), self._pre(real)
        loss = torch.tensor(0.0, device=fake.device)
        xf, xr = fp, rp
        for s in self.slices:
            xf = s(xf); xr = s(xr)
            loss = loss + F.l1_loss(xf, xr.detach())
        return loss


# ── Feature Matching ─────────────────────────────────────────────────────────
class FeatureMatchingLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, real_outs, fake_outs):
        loss = torch.tensor(0.0, device=real_outs[0][0].device)
        for (_, rf), (_, ff) in zip(real_outs, fake_outs):
            for r, f in zip(rf, ff):
                loss = loss + F.l1_loss(f, r.detach())
            loss = loss / len(rf)
        return loss / len(real_outs)


# ── AdaFace IR-50 Identity ───────────────────────────────────────────────────
def load_adaface_ir50(model_dir: str, device: str = 'cuda') -> nn.Module:
    """
    Load frozen AdaFace IR-50 from the downloaded HuggingFace folder.
    Uses net.py from /content/AdaFace + model.safetensors weights.
    All weights are frozen — never updated during training.
    """
    from safetensors.torch import load_file
    adaface_repo = '/content/AdaFace'
    if adaface_repo not in sys.path:
        sys.path.insert(0, adaface_repo)
    import net as adaface_net

    model = adaface_net.build_model('ir_50')
    state = load_file(os.path.join(model_dir, 'model.safetensors'), device='cpu')
    # Strip 'model.' prefix if present
    if any(k.startswith('model.') for k in state):
        state = {k[6:]: v for k, v in state.items() if k.startswith('model.')}
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:     print(f'  AdaFace missing keys   : {len(missing)}')
    if unexpected:  print(f'  AdaFace unexpected keys: {len(unexpected)}')
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    n = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'AdaFace IR-50 loaded and frozen ({n:.1f}M params)')
    return model.to(device)


class AdaFaceIdentityLoss(nn.Module):
    """
    Cosine distance in AdaFace IR-50 embedding space.

    Applied between generated photo and real photo ONLY.
    Active from epoch 1 so it shapes facial geometry during training.

    λ=10 (equal to VGG) so identity has enough signal to compete
    with L1 (100) and shape the encoder from the start.

    Loss: 0 = same person, 1 = unrelated, 2 = max
    """
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def _embed(self, x: torch.Tensor) -> torch.Tensor:
        x_112 = F.interpolate(x, size=(112, 112),
                              mode='bilinear', align_corners=False)
        out   = self.model(x_112)
        emb   = out[0] if isinstance(out, (tuple, list)) else out
        return F.normalize(emb, dim=1)

    def forward(self, fake: torch.Tensor, real: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            real_emb = self._embed(real)   # fixed reference
        fake_emb = self._embed(fake)       # gradients flow here
        return (1.0 - (fake_emb * real_emb).sum(dim=1)).mean()


print('All loss functions defined.')

All loss functions defined.


In [8]:
import random
from collections import deque

class ReplayBuffer:
    def __init__(self, max_size: int = 50):
        self.max_size = max_size
        self.buffer   = deque(maxlen=max_size)

    def query(self, images: torch.Tensor) -> torch.Tensor:
        result = []
        for img in images:
            img = img.unsqueeze(0)
            if len(self.buffer) < self.max_size:
                self.buffer.append(img.detach().clone())
                result.append(img)
            elif random.random() < 0.5:
                idx = random.randrange(len(self.buffer))
                stored = self.buffer[idx].clone()
                self.buffer[idx] = img.detach().clone()
                result.append(stored)
            else:
                self.buffer.append(img.detach().clone())
                result.append(img)
        return torch.cat(result, dim=0)

In [9]:
import torch
from pix2pix import UNetGenerator

torch.manual_seed(CFG['seed'])
device = torch.device('cuda')

# Models
G = init_weights(UNetGenerator(ngf=64)).to(device)
D = init_weights(MultiScaleDiscriminator()).to(device)

g_p = sum(p.numel() for p in G.parameters()) / 1e6
d_p = sum(p.numel() for p in D.parameters()) / 1e6
print(f'Generator            : {g_p:.2f}M params')
print(f'MultiScaleDisc       : {d_p:.2f}M params')

# Losses
lsgan_loss = LSGANLoss()
l1_loss    = torch.nn.L1Loss()
vgg_loss   = PerceptualLoss().to(device)
fm_loss    = FeatureMatchingLoss()
adaface_bb = load_adaface_ir50(ADAFACE_DIR, device=str(device))
id_loss    = AdaFaceIdentityLoss(adaface_bb)

# Optimisers
opt_G = torch.optim.Adam(G.parameters(), lr=CFG['lr'], betas=(CFG['beta1'], 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=CFG['lr'], betas=(CFG['beta1'], 0.999))

# LR schedule: flat -> linear decay to 0
def lr_lambda(epoch):
    if epoch < CFG['decay_start']: return 1.0
    n = max(1, CFG['n_epochs'] - CFG['decay_start'])
    return max(0.0, 1.0 - (epoch - CFG['decay_start']) / n)

sched_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_lambda)
sched_D = torch.optim.lr_scheduler.LambdaLR(opt_D, lr_lambda)

buffer = ReplayBuffer(max_size=50)
print('All components ready.')

Generator            : 54.41M params
MultiScaleDisc       : 5.54M params
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:07<00:00, 73.3MB/s]


  AdaFace missing keys   : 389
  AdaFace unexpected keys: 467
AdaFace IR-50 loaded and frozen (43.6M params)
All components ready.


In [10]:
with torch.no_grad():
    s = torch.randn(1, 3, 256, 256, device=device)
    p = torch.randn(1, 3, 256, 256, device=device)
    f = G(s)
    outs = D(s, p)

print(f'G output   : {f.shape}  [{f.min():.2f}, {f.max():.2f}]')
for i, (logits, feats) in enumerate(outs):
    print(f'D scale {i+1} : logits {logits.shape}  feats {[x.shape for x in feats]}')

# Test identity loss
fake_t = torch.randn(1, 3, 256, 256, device=device)
real_t = torch.randn(1, 3, 256, 256, device=device)
id_val = id_loss(fake_t, real_t)
print(f'AdaFace loss (random): {id_val.item():.4f}  (expect ~1.0)')

assert f.shape == (1, 3, 256, 256)
assert len(outs) == 2 and len(outs[0][1]) == 4
print('All checks passed.')

G output   : torch.Size([1, 3, 256, 256])  [-0.79, 0.80]
D scale 1 : logits torch.Size([1, 1, 30, 30])  feats [torch.Size([1, 64, 128, 128]), torch.Size([1, 128, 64, 64]), torch.Size([1, 256, 32, 32]), torch.Size([1, 512, 31, 31])]
D scale 2 : logits torch.Size([1, 1, 14, 14])  feats [torch.Size([1, 64, 64, 64]), torch.Size([1, 128, 32, 32]), torch.Size([1, 256, 16, 16]), torch.Size([1, 512, 15, 15])]
AdaFace loss (random): 0.2324  (expect ~1.0)
All checks passed.


In [11]:
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from dataset import CUHKDataset

train_ds = CUHKDataset(os.path.join(DATA_DIR, 'train'), augment=True)
test_ds  = CUHKDataset(os.path.join(DATA_DIR, 'test'),  augment=False)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          shuffle=True,  num_workers=CFG['num_workers'],
                          pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'],
                          pin_memory=True)

print(f'Train: {len(train_ds)} pairs | {len(train_loader)} batches')
print(f'Test : {len(test_ds)} pairs | {len(test_loader)} batches')

CUHKDataset | train | 150 pairs | augment=True
CUHKDataset | test | 38 pairs | augment=False
Train: 150 pairs | 37 batches
Test : 38 pairs | 10 batches


In [12]:
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm


def to_display(t):
    return (t * 0.5 + 0.5).clamp(0,1).cpu().permute(1,2,0).float().numpy()


def save_sample_grid(sketches, reals, fakes, epoch):
    n = min(4, sketches.size(0))
    fig, axes = plt.subplots(n, 3, figsize=(9, n*3))
    if n == 1: axes = axes[np.newaxis, :]
    for i in range(n):
        for j, (t, title) in enumerate([
            (sketches[i], 'Sketch'), (reals[i], 'Real'), (fakes[i], 'Generated')
        ]):
            axes[i,j].imshow(to_display(t))
            if i == 0: axes[i,j].set_title(title, fontsize=9)
            axes[i,j].axis('off')
    plt.suptitle(f'Epoch {epoch}', fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'samples', f'ep_{epoch:04d}.png'),
                dpi=100, bbox_inches='tight')
    plt.close()


def plot_losses(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(history['G'],   label='G',  color='tab:blue')
    axes[0].plot(history['D'],   label='D',  color='tab:orange')
    axes[0].set_title('GAN losses'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(history['l1'],  label='L1',  color='tab:green')
    axes[1].plot(history['vgg'], label='VGG', color='tab:purple')
    axes[1].plot(history['fm'],  label='FM',  color='tab:cyan')
    axes[1].set_title('Reconstruction'); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].plot(history['id'],  label='AdaFace (identity)', color='tab:red')
    axes[2].plot(history['adv'], label='G adv',              color='tab:gray')
    axes[2].set_title('Identity + adv'); axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curves.png'), dpi=100)
    plt.close()


def save_checkpoint(epoch, history):
    state = dict(epoch=epoch,
                 G=G.state_dict(), D=D.state_dict(),
                 opt_G=opt_G.state_dict(), opt_D=opt_D.state_dict(),
                 history=history)
    torch.save(state, os.path.join(OUTPUT_DIR, 'checkpoints', f'ckpt_{epoch:04d}.pth'))
    torch.save(state, os.path.join(OUTPUT_DIR, 'checkpoints', 'latest.pth'))


# Resume if checkpoint exists
start_epoch = 1
history     = {k: [] for k in ['G','D','adv','l1','vgg','fm','id']}

latest = os.path.join(OUTPUT_DIR, 'checkpoints', 'latest.pth')
if os.path.isfile(latest):
    ckpt        = torch.load(latest, map_location=device)
    G.load_state_dict(ckpt['G'])
    D.load_state_dict(ckpt['D'], strict=False)
    opt_G.load_state_dict(ckpt['opt_G'])
    opt_D.load_state_dict(ckpt['opt_D'])
    start_epoch = ckpt['epoch'] + 1
    history     = {k: ckpt['history'].get(k, []) for k in history}
    print(f'Resumed from epoch {ckpt["epoch"]}')
else:
    print('Training from scratch — epoch 1')

fixed_batch = next(iter(test_loader))
print(f'Training epochs {start_epoch} -> {CFG["n_epochs"]}')


# ── Main loop ─────────────────────────────────────────────────────────────────
for epoch in range(start_epoch, CFG['n_epochs'] + 1):
    G.train(); D.train()
    sums = {k: 0.0 for k in history}
    t0   = time.time()

    for batch in tqdm(train_loader,
                      desc=f'Ep {epoch:>3}/{CFG["n_epochs"]}',
                      leave=False):
        sketch = batch['sketch'].to(device)
        real   = batch['photo' ].to(device)

        fake = G(sketch)

        # Update D
        fake_buf     = buffer.query(fake)
        real_outs    = D(sketch, real)
        fake_outs    = D(sketch, fake_buf.detach())
        loss_D       = lsgan_loss.discriminator_loss(real_outs, fake_outs)
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # Update G
        fake_outs_g  = D(sketch, fake)
        real_outs_g  = D(sketch, real)

        loss_adv = lsgan_loss.generator_loss(fake_outs_g)
        loss_l1  = l1_loss(fake, real)              * CFG['lambda_l1']
        loss_vgg = vgg_loss(fake, real)             * CFG['lambda_vgg']
        loss_fm  = fm_loss(real_outs_g, fake_outs_g)* CFG['lambda_fm']
        loss_id  = id_loss(fake, real)              * CFG['lambda_id']
        loss_G   = loss_adv + loss_l1 + loss_vgg + loss_fm + loss_id

        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        sums['G']   += loss_G.item()
        sums['D']   += loss_D.item()
        sums['adv'] += loss_adv.item()
        sums['l1']  += loss_l1.item()  / CFG['lambda_l1']
        sums['vgg'] += loss_vgg.item() / CFG['lambda_vgg']
        sums['fm']  += loss_fm.item()  / CFG['lambda_fm']
        sums['id']  += loss_id.item()  / CFG['lambda_id']

    n = len(train_loader)
    for k in history: history[k].append(sums[k] / n)

    sched_G.step(); sched_D.step()
    lr_now = opt_G.param_groups[0]['lr']

    print(f'Ep {epoch:>3} | '
          f'G={history["G"][-1]:.4f}  D={history["D"][-1]:.4f}  '
          f'L1={history["l1"][-1]:.4f}  FM={history["fm"][-1]:.4f}  '
          f'ID={history["id"][-1]:.4f}  '
          f'lr={lr_now:.1e}  {time.time()-t0:.0f}s')

    if epoch % CFG['sample_every'] == 0 or epoch == 1:
        G.eval()
        with torch.no_grad():
            fakes = G(fixed_batch['sketch'].to(device))
        save_sample_grid(fixed_batch['sketch'], fixed_batch['photo'], fakes, epoch)
        G.train()

    if epoch % CFG['save_every'] == 0 or epoch == CFG['n_epochs']:
        save_checkpoint(epoch, history)

    if epoch % 20 == 0:
        plot_losses(history)

print('Training complete.')
plot_losses(history)

Training from scratch — epoch 1
Training epochs 1 -> 200


Ep   1/200:   0%|          | 0/37 [00:00<?, ?it/s]

Ep   1 | G=71.2603  D=0.4396  L1=0.3623  FM=0.1744  ID=0.1379  lr=2.0e-04  95s


Ep   2/200:   0%|          | 0/37 [00:00<?, ?it/s]

Ep   2 | G=55.9509  D=0.2147  L1=0.2506  FM=0.1066  ID=0.0783  lr=2.0e-04  14s


...

Ep 198 | G=35.5255  D=0.0043  L1=0.1315  FM=0.0870  ID=0.0308  lr=4.0e-06  15s


Ep 199/200:   0%|          | 0/37 [00:00<?, ?it/s]

Ep 199 | G=36.1924  D=0.0053  L1=0.1347  FM=0.0890  ID=0.0327  lr=2.0e-06  15s


Ep 200/200:   0%|          | 0/37 [00:00<?, ?it/s]

Ep 200 | G=36.9419  D=0.0036  L1=0.1415  FM=0.0896  ID=0.0348  lr=0.0e+00  15s
Training complete.
